In [0]:
# Bronze의 nested array 데이터를 개별 Silver 테이블로 정규화한다.
# 한 Study에 여러 Intervention, Outcome, Location이 존재하는 1:N 관계를 분리한다.

from pyspark.sql import functions as F

bronze_df = spark.read.table(
    "clinical_trials.bronze_studies"
)

silver_studies_df = spark.read.table(
    "clinical_trials.silver_studies"
)

silver_conditions_df = spark.read.table(
    "clinical_trials.silver_conditions"
)

print(bronze_df.count())

604566


In [0]:
# Interventions
# 원본 데이터 구조
# Study
#  └─ armsInterventionsModule
#      └─ interventions[]

# interventions 배열을 개별 row로 펼친다.
# posexplode_outer를 사용해 원본 배열 내 순서를 함께 보존한다.
# 이를 통해 intervention 종류별 집계 및 study와의 join이 쉬워진다.

interventions_exploded = (
    bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id"),

        # posexplode는 배열 내 순서까지 같이 줍니다.
        F.posexplode_outer(
            "study.protocolSection.armsInterventionsModule.interventions"
        ).alias("intervention_order", "intervention")
    )
)

silver_interventions_df = (
    interventions_exploded
    .filter(F.col("intervention").isNotNull())
    .select(
        "nct_id",
        "intervention_order",

        F.col("intervention.type")
            .alias("intervention_type"),

        F.col("intervention.name")
            .alias("intervention_name"),

        F.col("intervention.description")
            .alias("description"),

        F.col("intervention.armGroupLabels")
            .alias("arm_group_labels")
    )
)

display(silver_interventions_df.limit(30))

print("Intervention rows:", silver_interventions_df.count())

nct_id,intervention_order,intervention_type,intervention_name,description,arm_group_labels
NCT03705065,0,DRUG,Bupivacaine HCl,"Bupivacaine HCl without epinephrine 0.25%, 60 mL (30 mL per pectoral pocket)","List(Treatment Group 1, Treatment Group 2)"
NCT05519865,0,DRUG,Tucidinostat,30mg orally BIW,List(Tucidinostat Combined with Tislelizumab)
NCT05519865,1,DRUG,Tislelizumab,200 mg intravenously (IV) Q3W,"List(Tislelizumab, Tucidinostat Combined with Tislelizumab)"
NCT00622765,0,DRUG,placebo,Placebo capsule twice daily,List(004)
NCT00622765,1,DRUG,R256918,5 mg capsule twice daily,List(001)
NCT00622765,2,DRUG,R256918,10 mg capsule twice daily,List(002)
NCT00622765,3,DRUG,R256918,15 mg capsule twice daily,List(003)
NCT06935565,0,OTHER,Trifocal IOL implantation,Patients were implanted bilaterally with Asqelio Trifocal IOL 270±90 days before study visit,List(Trifocal)
NCT05371067,0,OTHER,Fructose respiratory test,Healthy volunteers will performed a fructose respiratory test in order to evaluate if they are or not fructose malabsorbing. The two population will be compared about their feelings and their micriobiote.,List(Healthy volunteers)
NCT01525667,0,BIOLOGICAL,150M PLX-PAD,"Single course, multiple IM injections",List(150M PLX-PAD)


Intervention rows: 1022647


In [0]:
# Intervention QC

# NCT ID null
silver_interventions_df.filter(
    F.col("nct_id").isNull()
).count()

# Composite key 중복
intervention_duplicates = (
    silver_interventions_df
    .groupBy("nct_id", "intervention_order")
    .count()
    .filter(F.col("count") > 1)
)

display(intervention_duplicates)

# Intervention Type 체크
display(
    silver_interventions_df
    .groupBy("intervention_type")
    .count()
    .orderBy(F.desc("count"))
)

nct_id,intervention_order,count


intervention_type,count
DRUG,415442
OTHER,181439
DEVICE,104808
BEHAVIORAL,99484
PROCEDURE,93481
BIOLOGICAL,47516
DIETARY_SUPPLEMENT,31115
DIAGNOSTIC_TEST,27493
RADIATION,13074
COMBINATION_PRODUCT,4723


In [0]:
(
    silver_interventions_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clinical_trials.silver_interventions")
)

In [0]:
# Outcomes
# 원본 구조는 서로 다른 두 배열인데, 분석 관점에서는 모두 Outcome입니다. -> Silver에서는 하나의 테이블로 합치기
#  outcomesModule
#  ├─ primaryOutcomes[]
#  └─ secondaryOutcomes[]

# ClinicalTrials.gov에서는 Primary/Secondary Outcome이 별도 배열로 제공된다.
# Silver에서는 공통 Outcome 개체로 보고 outcome_type 컬럼을 추가한 뒤
# unionByName으로 하나의 테이블로 통합한다.


# Primary Outcome
primary_outcomes_df = (
    bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id"),

        F.posexplode_outer(
            "study.protocolSection.outcomesModule.primaryOutcomes"
        ).alias("outcome_order", "outcome")
    )
    .filter(F.col("outcome").isNotNull())
    .select(
        "nct_id",
        F.lit("PRIMARY").alias("outcome_type"),
        "outcome_order",

        F.col("outcome.measure").alias("measure"),
        F.col("outcome.description").alias("description"),
        F.col("outcome.timeFrame").alias("time_frame")
    )
)

# Secondary Outcome
secondary_outcomes_df = (
    bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id"),

        F.posexplode_outer(
            "study.protocolSection.outcomesModule.secondaryOutcomes"
        ).alias("outcome_order", "outcome")
    )
    .filter(F.col("outcome").isNotNull())
    .select(
        "nct_id",
        F.lit("SECONDARY").alias("outcome_type"),
        "outcome_order",

        F.col("outcome.measure").alias("measure"),
        F.col("outcome.description").alias("description"),
        F.col("outcome.timeFrame").alias("time_frame")
    )
)

In [0]:
# unionByName : 같은 컬럼 구조를 가진 두 데이터셋의 row를 위아래로 붙이는 것
# 컬럼 순서보다 이름을 기준으로 맞춰주므로 일반 union보다 안전합니다.

silver_outcomes_df = primary_outcomes_df.unionByName(
    secondary_outcomes_df
)

display(silver_outcomes_df.limit(30))

print("Outcome rows:", silver_outcomes_df.count())

nct_id,outcome_type,outcome_order,measure,description,time_frame
NCT03705065,PRIMARY,0,Maximum Observed Plasma Concentration (Cmax),null,Baseline through 72 hours after start of study drug administration
NCT03705065,PRIMARY,1,Area Under the Concentration-time Curve From Time 0 to the Last Collection Time After Study Drug Administration (AUC Last),null,Baseline through 72 hours after start of study drug administration
NCT03705065,PRIMARY,2,Area Under the Concentration-time Curve From Time 0 to Infinity After Study Drug Administration (AUC∞),null,Baseline through 72 hours after start of study drug administration
NCT03705065,PRIMARY,3,Time to Maximum Plasma Concentration (Tmax),null,Baseline through 72 hours after start of study drug administration
NCT03705065,PRIMARY,4,The Apparent Terminal Elimination Rate Constant (λz),null,Baseline through 72 hours after start of study drug administration
NCT03705065,PRIMARY,5,The Apparent Terminal Elimination Half-life (t1/2el),null,Baseline through 72 hours after start of study drug administration
NCT05519865,PRIMARY,0,Progression Free Survival (PFS) per RECIST v1.1,"PFS assessed by investigator per RECIST v1.1, measured from the date of randomization until progression or death, whichever is first met.",Up to 2 years
NCT00622765,PRIMARY,0,Mean change in body weight from baseline to Week 12.,null,"Baseline, 12 weeks"
NCT06935565,PRIMARY,0,Residual refraction,Residual refractive error determined by subjective refraction procedures and expressed in diopters of spherical and cylindrical power,270±90 days after surgery
NCT06935565,PRIMARY,1,Light Distortion Index,"The distortion index is calculated as the ratio of the area of points missed by the subject and the total area explored using the Light Distortion Analyzer, and is expressed as a percentage",270±90 days from surgery


Outcome rows: 3567302


In [0]:
# Outcome QC

display(
    silver_outcomes_df
    .groupBy("outcome_type")
    .count()
)

# 중복 검사
outcome_duplicates = (
    silver_outcomes_df
    .groupBy(
        "nct_id",
        "outcome_type",
        "outcome_order"
    )
    .count()
    .filter(F.col("count") > 1)
)

display(outcome_duplicates)

# measure가 없는 데이터 확인
silver_outcomes_df.filter(
    F.col("measure").isNull()
).count()

outcome_type,count
PRIMARY,1190998
SECONDARY,2376304


nct_id,outcome_type,outcome_order,count


7

In [0]:
(
    silver_outcomes_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clinical_trials.silver_outcomes")
)

In [0]:
# Locations

# 원본 데이터 구조
# contactsLocationsModule
#  └─ locations[]

# locations 배열을 Study-Location 1:N 관계로 정규화한다.

locations_exploded = (
    bronze_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id"),

        F.posexplode_outer(
            "study.protocolSection.contactsLocationsModule.locations"
        ).alias("location_order", "location")
    )
)

silver_locations_df = (
    locations_exploded
    .filter(F.col("location").isNotNull())
    .select(
        "nct_id",
        "location_order",

        F.col("location.facility")
            .alias("facility"),

        F.col("location.status")
            .alias("location_status"),

        F.col("location.city")
            .alias("city"),

        F.col("location.state")
            .alias("state"),

        F.col("location.zip")
            .alias("zip"),

        F.col("location.country")
            .alias("country"),

        # 위도와 경도는 향후 위치 기반 분석이 가능하도록 double 타입으로 변환한다.
        F.col("location.geoPoint.lat")
            .cast("double")
            .alias("latitude"),

        F.col("location.geoPoint.lon")
            .cast("double")
            .alias("longitude")
    )
)

display(silver_locations_df.limit(30))

print("Location rows:", silver_locations_df.count())

nct_id,location_order,facility,location_status,city,state,zip,country,latitude,longitude
NCT03705065,0,Hermann Drive Surgical Hospital,null,Houston,Texas,77027,United States,29.76328,-95.36327
NCT05519865,0,Shanghai Pulmonary Hospital,null,Shanghai,Shanghai Municipality,200433,China,31.22222,121.45806
NCT00622765,0,null,null,Edegem,null,null,Belgium,51.15662,4.44504
NCT00622765,1,null,null,Liège,null,null,Belgium,50.63373,5.56749
NCT00622765,2,null,null,Frederiksberg C N/A,null,null,Denmark,null,null
NCT00622765,3,null,null,Hellerup,null,null,Denmark,55.73204,12.57093
NCT00622765,4,null,null,Vipperroed,null,null,Denmark,null,null
NCT00622765,5,null,null,Helsinki,null,null,Finland,60.16952,24.93545
NCT00622765,6,null,null,Kuopio,null,null,Finland,62.89238,27.67703
NCT00622765,7,null,null,Berlin,null,null,Germany,52.52437,13.41053


Location rows: 3532550


In [0]:
# Location QC
# 위치 데이터의 기본 domain validation을 수행한다.
# 위도는 -90~90, 경도는 -180~180 범위를 벗어나서는 안 된다.

# 국가 별 확인
display(
    silver_locations_df
    .groupBy("country")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

# 위도, 경도 검사 ( 위도는 -90 ~ 90, 경도는 -180 ~ 180 )
invalid_geo_df = silver_locations_df.filter(
    (F.col("latitude").isNotNull()) &
    (
        (F.col("latitude") < -90) |
        (F.col("latitude") > 90) |
        (F.col("longitude") < -180) |
        (F.col("longitude") > 180)
    )
)

display(invalid_geo_df)

country,count
United States,1499929
France,202340
China,173175
Germany,171615
Spain,118268
Japan,111425
Italy,108181
Canada,105799
United Kingdom,100386
Poland,59049


nct_id,location_order,facility,location_status,city,state,zip,country,latitude,longitude


In [0]:
(
    silver_locations_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clinical_trials.silver_locations")
)

In [0]:
# Referential Integrity QC
# 자식 테이블의 nct_id가 부모 silver_studies에 실제 존재하는지 검증한다.
# 부모가 없는 자식 row(orphan record)를 탐지하여 참조 무결성을 확인한다.

# 이전 Notebook에서 생성한 Python DataFrame 변수는 현재 Notebook에 공유되지 않는다.
# 따라서 저장된 Delta Table을 다시 읽어 현재 Spark session에서 사용할 DataFrame을 생성한다.
silver_studies_df = spark.read.table(
    "clinical_trials.silver_studies"
)

silver_conditions_df = spark.read.table(
    "clinical_trials.silver_conditions"
)

orphan_interventions = (
    silver_interventions_df
    .join(
        silver_studies_df.select("nct_id"),
        on="nct_id",
        how="left_anti"
    )
    # left_anti : 왼쪽에는 있는데 오른쪽에는 없는 row 찾기 -> Intervention에는 있는데 Study에는 없는 NCT ID
)

print("Orphan interventions:", orphan_interventions.count())

def orphan_count(child_df):
    return (
        child_df
        .join(
            silver_studies_df.select("nct_id"),
            on="nct_id",
            how="left_anti"
        )
        .count()
    )

print("Conditions:", orphan_count(silver_conditions_df))
print("Interventions:", orphan_count(silver_interventions_df))
print("Outcomes:", orphan_count(silver_outcomes_df))
print("Locations:", orphan_count(silver_locations_df))

Orphan interventions: 0
Conditions: 0
Interventions: 0
Outcomes: 0
Locations: 0


In [0]:
counts = {
    "studies": silver_studies_df.count(),
    "conditions": silver_conditions_df.count(),
    "interventions": silver_interventions_df.count(),
    "outcomes": silver_outcomes_df.count(),
    "locations": silver_locations_df.count(),
}

counts

{'studies': 604566,
 'conditions': 1087211,
 'interventions': 1022647,
 'outcomes': 3567302,
 'locations': 3532550}